In [1]:
import psycopg2
import pandas as pd

In [2]:
conn = psycopg2.connect(
    dbname="dbexam",
    user="postgres",
    password="DBmiko",
    host="localhost",
    port="5432"
)

In [3]:
# --- Query gabungan: assessment + attendance + enrollment ---
query = """
SELECT
    e.enroll_id,
    att.attendance_percentage,
    MAX(CASE WHEN ass.assessment_type = 'Quiz' THEN ass.score END) AS quiz_score,
    MAX(CASE WHEN ass.assessment_type = 'Midterm' THEN ass.score END) AS midterm_score,
    MAX(CASE WHEN ass.assessment_type = 'Project' THEN ass.score END) AS project_score,
    e.grade
FROM enrollment e
LEFT JOIN attendance att ON e.enroll_id = att.enroll_id
LEFT JOIN assessment ass ON e.enroll_id = ass.enroll_id
GROUP BY e.enroll_id, att.attendance_percentage, e.grade
HAVING COUNT(ass.assessment_id) >= 2
"""

In [4]:
df = pd.read_sql(query, conn)
conn.close()

C:\Users\ramda\AppData\Local\Temp\ipykernel_7684\2049290334.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [5]:
# --- Tampilkan preview dan cek baris kosong ---
print(f"Jumlah baris awal dari query: {len(df)}")
df.head()

Jumlah baris awal dari query: 2105


,enroll_id,attendance_percentage,quiz_score,midterm_score,project_score,grade
0,146,70,None,43,32,33
1,1615,87,None,86,80,85
2,1215,63,None,90,86,72
3,1093,82,None,80,54,66
4,916,97,None,96,62,71


In [6]:
# --- Cek data yang hilang (null) per kolom ---
print("Jumlah missing value per kolom:")
print(df.isnull().sum())

# --- Drop NA dan lihat hasil akhirnya ---
df_clean = df.dropna()
print(f"\nJumlah baris setelah dropna: {len(df_clean)}")
df_clean.head()


Jumlah missing value per kolom:
enroll_id                   0
attendance_percentage       0
quiz_score               2105
midterm_score               0
project_score               0
grade                       0
dtype: int64

Jumlah baris setelah dropna: 0


,enroll_id,attendance_percentage,quiz_score,midterm_score,project_score,grade


In [7]:
# Cek apakah cukup untuk training model
if len(df_clean) < 5:
    print("❌ Data terlalu sedikit untuk split dan training model.")
else:
    print("✅ Data cukup untuk lanjut training.")


❌ Data terlalu sedikit untuk split dan training model.
